# Classification metrics: ROC, PR curves, threshold choice

**Supervised learning.** Accuracy alone is not enough for imbalanced problems. This notebook plots **ROC** and **precision-recall** curves and shows how **threshold** moves the precision/recall tradeoff.

Uses `data/Bank_Personal_Loan_Modelling.csv` via `utils.data_paths`.

In [ ]:
import sys
from pathlib import Path

_repo = Path.cwd().resolve()
for _ in range(12):
    if (_repo / "utils" / "data_paths.py").exists():
        sys.path.insert(0, str(_repo))
        break
    if _repo.parent == _repo:
        raise FileNotFoundError("Run Jupyter from ml-notebook repo root.")
    _repo = _repo.parent

from utils.data_paths import data_dir
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    RocCurveDisplay,
    PrecisionRecallDisplay,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

df = pd.read_csv(data_dir() / "Bank_Personal_Loan_Modelling.csv")
X = df.drop(columns=["ID", "Personal Loan"])
y = df["Personal Loan"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, random_state=42)),
])
pipe.fit(X_train, y_train)
y_score = pipe.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_test, y_score, ax=ax[0], name="Logistic")
PrecisionRecallDisplay.from_predictions(y_test, y_score, ax=ax[1], name="Logistic")
plt.tight_layout()
plt.show()

best_t, best_f1 = 0.5, 0.0
for t in np.linspace(0.05, 0.95, 45):
    y_hat = (y_score >= t).astype(int)
    f1 = f1_score(y_test, y_hat)
    if f1 > best_f1:
        best_f1, best_t = f1, t
print("Best F1 on test grid:", round(best_f1, 4), "at threshold", round(best_t, 4))

t = 0.5
y_pred = (y_score >= t).astype(int)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["No loan", "Loan"])
disp.plot()
plt.title(f"Confusion matrix at threshold={t}")
plt.show()

## Try this

- Pick a threshold that maximizes **recall** for the positive class (loans) and report the precision cost.
- Plot confusion matrix at that threshold.